## Question1

The timestep t is a single integer, so it is turned into a vector of d numbers by evaluating sine and cosine waves of different frequencies at t. Low frequencies change slowly with t and high frequencies change quickly, so every t gets a unique, smooth "fingerprint" the UNet can read.

## Question2

In [1]:
import math, torch
d, max_period, t = 8, 10000, 1
half = d // 2
freqs = torch.exp(-math.log(max_period) * torch.arange(half) / half)
emb = torch.cat([torch.sin(t * freqs), torch.cos(t * freqs)])
print(emb)

tensor([0.8415, 0.0998, 0.0100, 0.0010, 0.5403, 0.9950, 0.9999, 1.0000])


## Question3

They use the same mathematical tool: a vector of sines and cosines at geometrically spaced frequencies (same formula as Q1, max period 10000). Both let a network read an integer (a position or a timestep) as a smooth, unique vector that it could not get from a single raw number. The key difference is what the integer means and where it is injected.

## Question4

8 × 8. Each stride-2 downsampling block halves the height and width: 64 → 32 → 16 → 8 (64 / 2^3 = 8).

## Question5

The UNet outputs its prediction of the noise that was added, ε_θ(x_t, t), with the same shape as x_t. During training we add known noise ε ~ N(0, I) to a clean image at a random step t, and the loss is the MSE between the true and predicted noise.

At generation time, the predicted noise is subtracted step by step to go from pure noise back to an image.

## Question6

a) dy/dx = 2x + 3 = 2(2) + 3 = 7. The code prints x.grad = tensor([7.]).
    
b) PyTorch builds no computational graph, so y has no grad_fn. Calling y.backward() raises RuntimeError: "element 0 of tensors does not require grad and does not have a grad_fn".

c) No. requires_grad defaults to False, so gradients are not tracked unless the flag is set.

## Question7

a) It prints w.grad = None. w was created without requires_grad=True, so autograd treats it as a constant and stores no gradient for it.

b) Create w with requires_grad=True. Then ∂y/∂w₀ = x² = 4 and ∂y/∂w₁ = x = 2:

In [7]:
x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([1.0, 3.0], requires_grad=True)
y = w[0] * x**2 + w[1] * x
y.backward()
print("x.grad =", x.grad)   
print("w.grad =", w.grad)  

x.grad = tensor([7.])
w.grad = tensor([4., 2.])


c) No, the default is requires_grad=False.

## Question8

detach() cuts z off from the graph, so z (and w) do not require grad and w.backward() raises an error. Fix: use clone(), which copies y but stays in the graph.

In [8]:
x = torch.tensor([1.0], requires_grad=True)
y = x * 3
z = y.clone()         
w = z * 2
w.backward()
print(x.grad)          

tensor([6.])


## Question9

PyTorch adds each new gradient to x.grad instead of replacing it. It prints 2 after the first backward, then 2 + 3 = 5 after the second. To avoid this, reset the gradient before each backward pass: call x.grad.zero_() (or optimizer.zero_grad() in a training loop), and the second result becomes 3.

In [10]:
x = torch.tensor([1.0], requires_grad=True)
y1 = x * 2
y1.backward()
print("After first backward: x.grad =", x.grad)   
x.grad.zero_()                                     
y2 = x * 3
y2.backward()
print("After second backward: x.grad =", x.grad)  

After first backward: x.grad = tensor([2.])
After second backward: x.grad = tensor([3.])
